# 03. Smoke test de integración A/B/C

Verificación funcional de que el pipeline de clasificación opera correctamente con las tres condiciones materializadas (A: imagen completa, B: ROI predicha, C: ROI GT).

**No son resultados experimentales.** Se entrena ResNet50 durante 1 sola época por condición para confirmar que toda la cadena funciona: dataset > dataloader > modelo > train > validate > evaluate > checkpoint > reload.

In [1]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
import torch
import torch.nn as nn


def locate_project_root() -> Path:
    current_dir = Path.cwd().resolve()
    for candidate in [current_dir, *current_dir.parents]:
        if (candidate / "src").exists() and (candidate / "data" / "processed" / "MMOTU_OTU_2D_experimental").exists():
            return candidate
    raise FileNotFoundError("No se pudo localizar la raíz del proyecto.")


PROJECT_ROOT = locate_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

CSV_PATH = PROJECT_ROOT / "configs" / "splits" / "mmotu_experimental_split_standard.csv"
DIR_A = PROJECT_ROOT / "data" / "processed" / "MMOTU_OTU_2D_experimental"
DIR_B = PROJECT_ROOT / "data" / "processed" / "MMOTU_OTU_2D_roi_pred"
DIR_C = PROJECT_ROOT / "data" / "processed" / "MMOTU_OTU_2D_roi_gt"
SAVE_DIR = PROJECT_ROOT / "results" / "classification" / "smoke_test_ABC"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
BATCH_SIZE = 32
LR = 1e-4
WEIGHT_DECAY = 1e-4
EPOCHS = 1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {device}")
print(f"Directorios:")
print(f"  A: {DIR_A}")
print(f"  B: {DIR_B}")
print(f"  C: {DIR_C}")
print(f"  Artefactos: {SAVE_DIR}")

Project root: /home/jvillanueva/projects/ovarian-tumor-ultrasound-classification
Device: cuda
Directorios:
  A: /home/jvillanueva/projects/ovarian-tumor-ultrasound-classification/data/processed/MMOTU_OTU_2D_experimental
  B: /home/jvillanueva/projects/ovarian-tumor-ultrasound-classification/data/processed/MMOTU_OTU_2D_roi_pred
  C: /home/jvillanueva/projects/ovarian-tumor-ultrasound-classification/data/processed/MMOTU_OTU_2D_roi_gt
  Artefactos: /home/jvillanueva/projects/ovarian-tumor-ultrasound-classification/results/classification/smoke_test_ABC


## 1. Class weights desde train

Se calculan una sola vez y se reusan en las tres condiciones.
Son idénticos porque las etiquetas provienen del mismo CSV.

In [2]:
from src.classification.dataset import MMOTUClassificationDataset
from src.classification.utils import compute_class_weights

ds_train_labels = MMOTUClassificationDataset( csv_file=str(CSV_PATH), image_dir=str(DIR_A), split="train", condition="A", )

train_labels = ds_train_labels.df["binary_label"].values
class_weights = compute_class_weights(train_labels).to(device)

print(f"Train samples: {len(train_labels)}")
print(f"Class distribution: {dict(zip(*np.unique(train_labels, return_counts=True)))}")
print(f"Class weights: {class_weights}")

Train samples: 800
Class distribution: {np.int64(0): np.int64(713), np.int64(1): np.int64(87)}
Class weights: tensor([0.5610, 4.5977], device='cuda:0')


## 2. Función de smoke test

Ejecuta 1 época de entrenamiento + evaluación en validation para una condición dada.
Verifica guardado y recarga del checkpoint.

In [3]:
from src.classification.transforms import get_train_transforms, get_val_transforms
from src.classification.dataloader import create_dataloader
from src.classification.models import create_resnet50_classifier
from src.classification.train import set_seed, train_model
from src.classification.evaluate import predict, evaluate_predictions


def run_smoke_test(condition, image_dir, save_dir, class_weights, device, seed=42):
    """Ejecuta 1 época de ResNet50 para una condición y reporta métricas."""
    print(f"\n{'='*60}")
    print(f"SMOKE TEST — Condición {condition}")
    print(f"{'='*60}")

    set_seed(seed)

    # Datasets
    ds_train = MMOTUClassificationDataset(
        csv_file=str(CSV_PATH), image_dir=str(image_dir),
        split="train", transform=get_train_transforms(), condition=condition,
    )
    ds_val = MMOTUClassificationDataset(
        csv_file=str(CSV_PATH), image_dir=str(image_dir),
        split="validation", transform=get_val_transforms(), condition=condition,
    )
    print(f"  Train: {len(ds_train)} | Validation: {len(ds_val)}")

    # Dataloaders
    train_loader = create_dataloader(ds_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, seed=seed)
    val_loader = create_dataloader(ds_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, seed=seed)

    # Modelo
    set_seed(seed)
    model = create_resnet50_classifier(pretrained=True, num_classes=2, freeze_backbone=False).to(device)

    # Criterion y optimizer
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    # Checkpoint path
    save_path = save_dir / f"resnet50_{condition}_best.pth"

    # Train 1 epoch
    history = train_model(
        model=model, train_loader=train_loader, val_loader=val_loader,
        criterion=criterion, optimizer=optimizer, device=device,
        epochs=EPOCHS, save_path=str(save_path),
        patience=5, min_delta=0.001, seed=seed, use_amp=True,
    )

    # Evaluate en validation set
    y_true, y_pred, y_prob = predict(model, val_loader, device)
    metrics = evaluate_predictions(y_true, y_pred, y_prob)

    # Verificación de checkpoint y meta
    checkpoint_exists = save_path.exists()
    meta_path = save_path.with_name(save_path.stem + "_meta.json")
    meta_exists = meta_path.exists()

    reload_ok = False
    if checkpoint_exists:
        test_model = create_resnet50_classifier(pretrained=False, num_classes=2).to(device)
        test_model.load_state_dict(torch.load(str(save_path), map_location=device, weights_only=True))
        reload_ok = True

    meta_content = None
    if meta_exists:
        with open(meta_path) as f:
            meta_content = json.load(f)

    result = {
        "condition": condition,
        "train_loss": history["train_loss"][-1],
        "val_weighted_loss": history["val_loss"][-1],
        "val_unweighted_loss": history["val_unweighted_loss"][-1],
        "val_accuracy": metrics["accuracy"],
        "average_precision": metrics["average_precision"],
        "recall_malignant": metrics["recall_malignant"],
        "balanced_accuracy": metrics["balanced_accuracy"],
        "checkpoint_saved": checkpoint_exists,
        "meta_saved": meta_exists,
        "checkpoint_reloadable": reload_ok,
    }

    print(f"\n  Resultados condición {condition}:")
    print(f"    Train loss:          {result['train_loss']:.4f}")
    print(f"    Val weighted loss:   {result['val_weighted_loss']:.4f}")
    print(f"    Val unweighted loss: {result['val_unweighted_loss']:.4f}")
    print(f"    Val accuracy:        {result['val_accuracy']:.4f}")
    print(f"    Average Precision:   {result['average_precision']:.4f}")
    print(f"    Recall maligno:      {result['recall_malignant']:.4f}")
    print(f"    Balanced Accuracy:   {result['balanced_accuracy']:.4f}")
    print(f"    Checkpoint guardado: {result['checkpoint_saved']}")
    print(f"    Meta guardado:       {result['meta_saved']}")
    print(f"    Checkpoint recargable: {result['checkpoint_reloadable']}")
    if meta_content:
        print(f"    Meta: {meta_content}")

    return result


print("Función run_smoke_test definida.")

Función run_smoke_test definida.


## 3. Smoke test - Condición A (imagen completa)

In [4]:
result_a = run_smoke_test("A", DIR_A, SAVE_DIR, class_weights, device, seed=SEED)


SMOKE TEST — Condición A
  Train: 800 | Validation: 200
Epoch 1/1


/home/jvillanueva/projects/ovarian-tumor-ultrasound-classification/src/classification/train.py:127: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if _use_amp else None
/home/jvillanueva/projects/ovarian-tumor-ultrasound-classification/src/classification/train.py:40: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  Train Loss: 0.6870 | Train Acc: 0.7850 | Val Loss(w): 0.6491 | Val Loss(uw): 0.6152 | Val Acc: 0.5800
  Modelo guardado

  Resultados condición A:
    Train loss:          0.6870
    Val weighted loss:   0.6491
    Val unweighted loss: 0.6152
    Val accuracy:        0.5800
    Average Precision:   0.3047
    Recall maligno:      0.5455
    Balanced Accuracy:   0.5649
    Checkpoint guardado: True
    Meta guardado:       True
    Checkpoint recargable: True
    Meta: {'epoch': 1, 'val_weighted_loss': 0.649080502986908, 'val_unweighted_loss': 0.6152313327789307, 'val_acc': 0.58}


## 4. Smoke test - Condición B (ROI predicha)

In [5]:
result_b = run_smoke_test("B", DIR_B, SAVE_DIR, class_weights, device, seed=SEED)


SMOKE TEST — Condición B
  Train: 800 | Validation: 200
Epoch 1/1


/home/jvillanueva/projects/ovarian-tumor-ultrasound-classification/src/classification/train.py:127: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if _use_amp else None
/home/jvillanueva/projects/ovarian-tumor-ultrasound-classification/src/classification/train.py:40: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  Train Loss: 0.6725 | Train Acc: 0.7538 | Val Loss(w): 0.6797 | Val Loss(uw): 0.6351 | Val Acc: 0.6750
  Modelo guardado

  Resultados condición B:
    Train loss:          0.6725
    Val weighted loss:   0.6797
    Val unweighted loss: 0.6351
    Val accuracy:        0.6750
    Average Precision:   0.1728
    Recall maligno:      0.4091
    Balanced Accuracy:   0.5585
    Checkpoint guardado: True
    Meta guardado:       True
    Checkpoint recargable: True
    Meta: {'epoch': 1, 'val_weighted_loss': 0.6797151136398315, 'val_unweighted_loss': 0.6350518727302551, 'val_acc': 0.675}


## 5. Smoke test - Condición C (ROI GT)

In [6]:
result_c = run_smoke_test("C", DIR_C, SAVE_DIR, class_weights, device, seed=SEED)


SMOKE TEST — Condición C
  Train: 800 | Validation: 200
Epoch 1/1


/home/jvillanueva/projects/ovarian-tumor-ultrasound-classification/src/classification/train.py:127: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if _use_amp else None
/home/jvillanueva/projects/ovarian-tumor-ultrasound-classification/src/classification/train.py:40: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  Train Loss: 0.6820 | Train Acc: 0.7800 | Val Loss(w): 0.6751 | Val Loss(uw): 0.5911 | Val Acc: 0.8100
  Modelo guardado

  Resultados condición C:
    Train loss:          0.6820
    Val weighted loss:   0.6751
    Val unweighted loss: 0.5911
    Val accuracy:        0.8100
    Average Precision:   0.3045
    Recall maligno:      0.4091
    Balanced Accuracy:   0.6343
    Checkpoint guardado: True
    Meta guardado:       True
    Checkpoint recargable: True
    Meta: {'epoch': 1, 'val_weighted_loss': 0.6751366186141968, 'val_unweighted_loss': 0.5910780096054077, 'val_acc': 0.81}


## 6. Resumen comparativo A/B/C

Estos valores son solo de verificación funcional. No deben interpretarse como resultados experimentales (1 sola época, sin convergencia).

In [7]:
summary_df = pd.DataFrame([result_a, result_b, result_c])
summary_df = summary_df.set_index("condition")

print("Resumen smoke test A/B/C (1 época, ResNet50, verificación funcional)")
print("=" * 70)
display(summary_df.T)

# Verificaciones finales
all_checkpoints = all([
    result_a["checkpoint_saved"],
    result_b["checkpoint_saved"],
    result_c["checkpoint_saved"],
])
all_reloadable = all([
    result_a["checkpoint_reloadable"],
    result_b["checkpoint_reloadable"],
    result_c["checkpoint_reloadable"],
])

print(f"\nTodos los checkpoints guardados: {all_checkpoints}")
print(f"Todos los checkpoints recargables: {all_reloadable}")
print(f"\nArtefactos en: {SAVE_DIR}")
for p in sorted(SAVE_DIR.iterdir()):
    print(f"  {p.name} ({p.stat().st_size / 1024:.0f} KB)")

Resumen smoke test A/B/C (1 época, ResNet50, verificación funcional)


condition,A,B,C
train_loss,0.68695,0.672533,0.681994
val_weighted_loss,0.649081,0.679715,0.675137
val_unweighted_loss,0.615231,0.635052,0.591078
val_accuracy,0.58,0.675,0.81
average_precision,0.304719,0.172785,0.304469
recall_malignant,0.545455,0.409091,0.409091
balanced_accuracy,0.564862,0.558478,0.634321
checkpoint_saved,True,True,True
meta_saved,True,True,True
checkpoint_reloadable,True,True,True



Todos los checkpoints guardados: True
Todos los checkpoints recargables: True

Artefactos en: /home/jvillanueva/projects/ovarian-tumor-ultrasound-classification/results/classification/smoke_test_ABC
  resnet50_A_best.pth (92157 KB)
  resnet50_A_best_meta.json (0 KB)
  resnet50_B_best.pth (92157 KB)
  resnet50_B_best_meta.json (0 KB)
  resnet50_C_best.pth (92157 KB)
  resnet50_C_best_meta.json (0 KB)
